In [8]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

BASE_YEAR = 2008  # Eurostat via FRED base

df = (pd.read_csv("../Data/CLVMNACSCAB1GQEL.csv")
        .rename(columns={"observation_date": "date", "CLVMNACSCAB1GQEL": "value"}))
df["date"] = pd.to_datetime(df["date"])
df = df[["date", "value"]].dropna().sort_values("date")

base = df.loc[df["date"].dt.year == BASE_YEAR, "value"].mean()
df["index"] = df["value"] / base * 100
df["series"] = "gdp"

rule_100 = alt.Chart(pd.DataFrame({"y": [100]})).mark_rule(
    color="#cbd5e1", strokeDash=[1, 5], strokeWidth=1).encode(y="y:Q")

line = alt.Chart(df).mark_line(strokeWidth=1.6).encode(
    x=alt.X("date:T", axis=alt.Axis(format="%Y", tickCount=8), title=None),
    y=alt.Y("index:Q", scale=alt.Scale(zero=False), title="Real GDP (2008 = 100)"),
    color=alt.Color("series:N", legend=None))

caption = alt.Title(
    text="Source: Eurostat via FRED®",
    subtitle=[
        "Real GDP, chain-linked volumes, indexed to 2008 = 100. Quarterly, 1995–2026.",
        "Still around 13% below its pre-crisis peak, seventeen years on.",
    ],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (
    (rule_100 + line)
    .properties(width=700, height=280, title=caption)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent")
    .configure_axis(labelColor="#676A86", titleColor="#676A86"))

styles.save(chart, name="greece_GDP", svg=True)
chart

alt.LayerChart(...)